<a href="https://colab.research.google.com/github/Vdmtx/FBNeo-Android/blob/main/C%C3%B3pia_de_Gerador_de_Imagem_15_07_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CÉLULA 1: INSTALAÇÃO E CARREGAMENTO DO MODELO
print(">> Bloco de Setup Único: Iniciando a instalação e o carregamento do modelo...")
print(">> Por favor, aguarde. Este processo pode demorar alguns minutos.")

# 1. INSTALAÇÃO MÍNIMA
!pip install diffusers transformers accelerate gradio --quiet
print(">> Bibliotecas instaladas.")

import torch
from diffusers import StableDiffusionPipeline
import gradio as gr

# 2. CARREGAR O MODELO ESPECIALISTA EM FOTORREALISMO
model_id = "digiplay/AbsoluteReality_v1.8.1"

print(f">> A carregar o modelo '{model_id}' para a memória da GPU...")
# O 'safety_checker' é desativado para poupar memória.
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16, safety_checker=None)
pipe.to("cuda")

# A otimização de memória é sempre uma boa prática.
pipe.enable_model_cpu_offload()

print("\n✅ Setup Concluído! O modelo está carregado e pronto na memória.")
print("⬇️ Agora, pode executar a CÉLULA 2 para abrir o painel de controle interativo.")

>> Bloco de Setup Único: Iniciando a instalação e o carregamento do modelo...
>> Por favor, aguarde. Este processo pode demorar alguns minutos.
>> Bibliotecas instaladas.
>> A carregar o modelo 'digiplay/AbsoluteReality_v1.8.1' para a memória da GPU...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/clip/feature_extraction_clip.py:30: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(



✅ Setup Concluído! O modelo está carregado e pronto na memória.
⬇️ Agora, pode executar a CÉLULA 2 para abrir o painel de controle interativo.


In [ ]:
# CÉLULA 2: PAINEL DE CONTROLE INTERATIVO (GRADIO)

# 1. DEFINIR A FUNÇÃO DE GERAÇÃO
# Esta função pega nos valores da interface e gera a imagem.
def gerar_imagem(prompt, negative_prompt, altura, largura, passos, cfg, semente):

    # Garante que as dimensões são múltiplas de 8, o que é ideal para o Stable Diffusion
    altura = (altura // 8) * 8
    largura = (largura // 8) * 8

    print(f"Gerando imagem com as dimensões: {largura}x{altura}")
    print(f"Prompt: {prompt}")

    # Usa -1 para uma semente aleatória
    if semente == -1:
        generator = None
    else:
        generator = torch.Generator(device="cuda").manual_seed(int(semente))

    # Gera a imagem com os parâmetros recebidos da interface
    imagem_gerada = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=altura,
        width=largura,
        num_inference_steps=passos,
        guidance_scale=cfg,
        generator=generator
    ).images[0]

    return imagem_gerada

# 2. CONSTRUIR A INTERFACE COM GRADIO
with gr.Blocks(css="footer {display: none !important}", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎨 Estúdio de Criação de Atores Virtuais")
    gr.Markdown("Insira os prompts e ajuste os controles para gerar o seu ator fotorrealista.")

    with gr.Row():
        with gr.Column(scale=2):
            # Controles de texto
            prompt_input = gr.Textbox(label="Prompt (O que você quer ver)", value="foto de rosto, (obra-prima), melhor qualidade, fotorrealista, 8k, mulher de 35 anos, cabelo ruivo, sardas, olhos verdes, expressão séria, textura de pele detalhada", lines=4)
            negative_prompt_input = gr.Textbox(label="Prompt Negativo (O que você NÃO quer ver)", value="desenho, pintura, 3d, (deformado, má qualidade:1.4), pele de plástico, artificial, simétrico demais", lines=2)

            # Controles de Tamanho
            with gr.Row():
                altura_input = gr.Slider(minimum=512, maximum=1024, step=8, value=768, label="Altura da Imagem")
                largura_input = gr.Slider(minimum=512, maximum=1024, step=8, value=512, label="Largura da Imagem")

            # Controles de Qualidade
            with gr.Row():
                passos_input = gr.Slider(minimum=15, maximum=100, step=1, value=30, label="Passos de Inferência (Qualidade)")
                cfg_input = gr.Slider(minimum=1.0, maximum=20.0, step=0.5, value=7.5, label="Escala de Guiamento (Fidelidade ao prompt)")
                semente_input = gr.Number(label="Semente (Seed) (-1 para aleatório)", value=-1)

            # Botão de Ação
            btn_gerar = gr.Button("Gerar Imagem", variant="primary")

        with gr.Column(scale=1):
            # Área de Resultado
            imagem_output = gr.Image(label="Ator Gerado", type="pil")

    # Conecta o botão à função
    btn_gerar.click(
        fn=gerar_imagem,
        inputs=[prompt_input, negative_prompt_input, altura_input, largura_input, passos_input, cfg_input, semente_input],
        outputs=imagem_output
    )

# 3. LANÇAR A INTERFACE
demo.launch(debug=True)

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4d1d68a10930199dbe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gerando imagem com as dimensões: 512x768
Prompt: foto de rosto, (obra-prima), melhor qualidade, fotorrealista, 8k, mulher de 35 anos, cabelo ruivo, sardas, olhos verdes, expressão séria, textura de pele detalhada


  0%|          | 0/30 [00:00<?, ?it/s]

Gerando imagem com as dimensões: 512x768
Prompt: foto de rosto, (obra-prima), melhor qualidade, fotorrealista, 8k, mulher de 35 anos, cabelo ruivo, sardas, olhos verdes, expressão séria, textura de pele detalhada


  0%|          | 0/52 [00:00<?, ?it/s]

Gerando imagem com as dimensões: 512x768
Prompt: foto de rosto, (obra-prima), melhor qualidade, fotorrealista, 8k, mulher de 35 anos, cabelo ruivo, sardas, olhos verdes, expressão séria, textura de pele detalhada


  0%|          | 0/86 [00:00<?, ?it/s]